In [ ]:
import sys, platform, hashlib, datetime, zoneinfo

NAMA = 'Femas Akbar Faturrohim'
NIM = '244107020200'

N     = int(NIM[-3:])
PARAM = {
    'bins' : 2 ** ((N % 4) + 3),
    'clip' : round(1.0 + (N % 5) * 0.5, 1),
    'tile' : (N % 4) + 2,
    'L'    : 2 ** ((N % 3) + 1),
}
waktu = datetime.datetime.now(zoneinfo.ZoneInfo('Asia/Jakarta'))

print('NAMA   :', NAMA)
print('NIM    :', NIM, '| N =', N)
for k, v in PARAM.items():
    print('%-6s :' % k, v)
print('WAKTU :', waktu.strftime('%Y-%m-%d %H:%M:%S %Z'))
print('SISTEM :', sys.version.split()[0], platform.platform())
kunci = NIM + '|' + waktu.strftime('%Y-%m-%d')
print('TOKEN :', hashlib.sha256(kunci.encode()).hexdigest()[:16])


## Persiapan Lingkungan dan Direktori
Mount Google Drive dan inisialisasi path direktori sesuai ketentuan pada modul praktikum.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CONTOH = '/content/drive/MyDrive/PCVK/Images'
BASE   = '/content/drive/MyDrive/PCVK/Images/' + NIM


### Langkah 2 — Import Library
Import library yang digunakan selama praktikum.


In [ ]:
import cv2 as cv
from google.colab.patches import cv2_imshow
from skimage import io
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import glob


---
# E-1 PERCOBAAN HISTOGRAM

### Langkah 3 — Membuat Histogram Citra Berdasarkan Flowchart
Menghitung frekuensi kemunculan intensitas piksel secara manual pada kanal R, G, dan B.

#### Tahap 1: Citra Contoh (`lena.jpg`)


In [ ]:
img = cv.imread(CONTOH + '/lena.jpg')
img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

height, width, depth = np.shape(img)
names = np.arange(256)

red = [0] * 256
green = [0] * 256
blue = [0] * 256

for y in range(0, height):
    for x in range(0, width):
        red[img[y][x][0]] += 1
        green[img[y][x][1]] += 1
        blue[img[y][x][2]] += 1

names = np.arange(256)
fig, axs = plt.subplots(1, 3, figsize=[20, 5], sharex=True, sharey=True)
fig.suptitle(NIM + ' - Histogram RGB plot (lena.jpg)')
fig.text(0.09, 0.5, 'Jumlah Kemunculan', va='center', rotation='vertical')
fig.text(0.5, 0.04, 'Intensitas Warna', ha='center')
axs[0].bar(names, red, color='red')
axs[1].bar(names, green, color='green')
axs[2].bar(names, blue, color='blue')
plt.show()


#### Tahap 2: Penerapan pada Citra KTM (`KTM1b.jpg`)


In [ ]:
img_ktm = cv.imread(BASE + '/KTM1b.jpg')
img_ktm = cv.cvtColor(img_ktm, cv.COLOR_BGR2RGB)

height, width, depth = np.shape(img_ktm)
red = [0] * 256
green = [0] * 256
blue = [0] * 256

for y in range(0, height):
    for x in range(0, width):
        red[img_ktm[y][x][0]] += 1
        green[img_ktm[y][x][1]] += 1
        blue[img_ktm[y][x][2]] += 1

names = np.arange(256)
fig, axs = plt.subplots(1, 3, figsize=[20, 5], sharex=True, sharey=True)
fig.suptitle(NIM + ' - Histogram RGB plot (KTM1b.jpg)')
fig.text(0.09, 0.5, 'Jumlah Kemunculan', va='center', rotation='vertical')
fig.text(0.5, 0.04, 'Intensitas Warna', ha='center')
axs[0].bar(names, red, color='red')
axs[1].bar(names, green, color='green')
axs[2].bar(names, blue, color='blue')
plt.show()


### Pertanyaan Praktikum E1 — Nomor 1
Membuat histogram menggunakan `numpy.histogram` dan `cv.calcHist`, lalu membandingkan ketiganya dengan menghitung selisih maksimum (harus bernilai 0).


In [ ]:
def histogram_manual(gray, L=256):
    h = np.zeros(L, dtype=np.int64)
    for y in range(gray.shape[0]):
        for x in range(gray.shape[1]):
            h[gray[y, x]] += 1
    return h

def periksa_histogram(path_citra, label_citra):
    gray = cv.imread(path_citra, cv.IMREAD_GRAYSCALE)
    
    # 1. Manual
    h_manual = histogram_manual(gray)
    
    # 2. NumPy histogram
    h_numpy, _ = np.histogram(gray.ravel(), bins=256, range=[0, 256])
    
    # 3. OpenCV calcHist
    h_cv = cv.calcHist([gray], [0], None, [256], [0, 256]).flatten().astype(np.int64)
    
    # Selisih maksimum
    diff_manual_numpy = np.max(np.abs(h_manual - h_numpy))
    diff_manual_cv = np.max(np.abs(h_manual - h_cv))
    diff_numpy_cv = np.max(np.abs(h_numpy - h_cv))
    
    print(f"[{label_citra}]")
    print("Selisih max Manual vs NumPy :", diff_manual_numpy)
    print("Selisih max Manual vs OpenCV:", diff_manual_cv)
    print("Selisih max NumPy vs OpenCV :", diff_numpy_cv)
    print("Apakah semua selisih 0?     :", (diff_manual_numpy == 0 and diff_manual_cv == 0 and diff_numpy_cv == 0))
    print("-" * 50)

periksa_histogram(CONTOH + '/lena.jpg', 'lena.jpg')
periksa_histogram(BASE + '/KTM1b.jpg', 'KTM1b.jpg')


**Jawaban E1 - No. 1:**
Selisih maksimum antar ketiga metode bernilai 0 karena ketiga metode sama-sama menghitung frekuensi kemunculan tiap nilai piksel secara diskrit tanpa aproksimasi.


### Pertanyaan Praktikum E1 — Nomor 2
Menggambarkan histogram ternormalisasi $p(j)$ dan kurva CDF untuk `KTM1a.jpg` s.d. `KTM1d.jpg` dalam layout $2 \times 2$.


In [ ]:
citra_files = ['KTM1a.jpg', 'KTM1b.jpg', 'KTM1c.jpg', 'KTM1d.jpg']
keterangan  = ['KTM1a (Ruang Gelap)', 'KTM1b (Lampu Ruangan)', 'KTM1c (Flash Kamera)', 'KTM1d (Cahaya Matahari)']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (berkas, ket) in enumerate(zip(citra_files, keterangan)):
    path = os.path.join(BASE, berkas)
    gray = cv.imread(path, cv.IMREAD_GRAYSCALE)
    
    if gray is not None:
        h = histogram_manual(gray)
        p = h / gray.size
        cdf = np.cumsum(p)
        
        ax = axes[idx]
        ax.plot(p, color='blue', label='p(j) Normalisasi')
        ax.plot(cdf, color='red', label='CDF')
        ax.set_title(f"{NIM} - {ket}")
        ax.set_xlabel('Intensitas Piksel')
        ax.set_ylabel('Probabilitas Kumulatif')
        ax.legend(loc='center right')
        ax.grid(True, linestyle='--', alpha=0.5)
    else:
        axes[idx].set_title(f"{ket} (File tidak ditemukan)")

plt.tight_layout()
plt.show()


**Jawaban E1 - No. 2:**
- **Citra paling gelap (KTM1a.jpg):** Kurva CDF melonjak sangat curam di bagian awal (intensitas rendah) karena mayoritas piksel bernilai gelap akibat minimnya pencahayaan saat akuisisi di ruangan gelap.
- **Citra paling terang (KTM1d.jpg / KTM1c.jpg):** Kurva CDF lebih landai di awal dan baru naik tajam di nilai intensitas tinggi karena pencahayaan matahari/flash mendominasi area foto.


### Pertanyaan Praktikum E1 — Nomor 3
Histogram `KTM1b.jpg` dengan jumlah bin sebesar `PARAM['bins']` (NIM 244107020200 = 8 bin) vs 256 bin.


In [ ]:
gray_b = cv.imread(BASE + '/KTM1b.jpg', cv.IMREAD_GRAYSCALE)
bins_val = PARAM['bins']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram Bins Pribadi (8 bin)
axes[0].hist(gray_b.ravel(), bins=bins_val, range=[0, 256], color='teal', edgecolor='black')
axes[0].set_title(f"{NIM} - Histogram KTM1b ({bins_val} Bins)")
axes[0].set_xlabel('Intensitas')
axes[0].set_ylabel('Jumlah Piksel')

# Histogram 256 Bins
axes[1].hist(gray_b.ravel(), bins=256, range=[0, 256], color='salmon', edgecolor='black')
axes[1].set_title(f"{NIM} - Histogram KTM1b (256 Bins)")
axes[1].set_xlabel('Intensitas')
axes[1].set_ylabel('Jumlah Piksel')

plt.tight_layout()
plt.show()


**Jawaban E1 - No. 3:**
Ketika bin dikurangi dari 256 menjadi 8 bin, informasi sebaran detail nilai intensitas piksel hilang. Variasi halus dan puncak sempit intensitas lokal tergeneralisasi karena tiap bin mencakup rentang 32 level keabuan.


---
# E-2 PERCOBAAN HISTOGRAM EQUALIZATION

### 1. Histogram Equalization Manual
Berdasarkan flowchart modul: frekuensi, penjumlahan kumulatif (CDF), normalisasi ke skala warna $K_0$, dan pemetaan ke citra baru.


In [ ]:
def he_manual(img_gray):
    # 1. Hitung frekuensi piksel
    h = np.zeros(256, dtype=np.int64)
    for y in range(img_gray.shape[0]):
        for x in range(img_gray.shape[1]):
            h[img_gray[y, x]] += 1
            
    # 2. Penjumlahan kumulatif dibagi jumlah piksel (CDF)
    total_pixels = img_gray.size
    cdf = np.cumsum(h) / total_pixels
    
    # 3. Implementasi rumus Ko = round(CDF * 255)
    ko = np.round(cdf * 255).astype(np.uint8)
    
    # 4. Transformasi kembali ke bentuk citra
    img_equalized = ko[img_gray]
    return img_equalized


#### Tahap 1: Verifikasi pada `lena_lc.jpg`


In [ ]:
lena_lc = cv.imread(CONTOH + '/lena_lc.jpg', cv.IMREAD_GRAYSCALE)
lena_lc_he = he_manual(lena_lc)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].imshow(lena_lc, cmap='gray')
axes[0, 0].set_title(NIM + ' - lena_lc.jpg Asli')
axes[0, 0].axis('off')

axes[0, 1].imshow(lena_lc_he, cmap='gray')
axes[0, 1].set_title(NIM + ' - lena_lc.jpg HE Manual')
axes[0, 1].axis('off')

axes[1, 0].hist(lena_lc.ravel(), 256, [0, 256], color='gray')
axes[1, 0].set_title('Histogram Asli')

axes[1, 1].hist(lena_lc_he.ravel(), 256, [0, 256], color='black')
axes[1, 1].set_title('Histogram Hasil HE Manual')

plt.tight_layout()
plt.show()


#### Tahap 2: Penerapan pada `KTM1a.jpg` (Ruang Gelap)


In [ ]:
ktm_a = cv.imread(BASE + '/KTM1a.jpg', cv.IMREAD_GRAYSCALE)
ktm_a_he = he_manual(ktm_a)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].imshow(ktm_a, cmap='gray')
axes[0, 0].set_title(NIM + ' - KTM1a.jpg Asli')
axes[0, 0].axis('off')

axes[0, 1].imshow(ktm_a_he, cmap='gray')
axes[0, 1].set_title(NIM + ' - KTM1a.jpg HE Manual')
axes[0, 1].axis('off')

axes[1, 0].hist(ktm_a.ravel(), 256, [0, 256], color='gray')
axes[1, 0].set_title('Histogram KTM1a Asli')

axes[1, 1].hist(ktm_a_he.ravel(), 256, [0, 256], color='black')
axes[1, 1].set_title('Histogram KTM1a HE Manual')

plt.tight_layout()
plt.show()


### 2. Histogram Equalization Menggunakan `cv.equalizeHist`
Membandingkan hasil `equalizeHist` dengan hasil implementasi manual dan menghitung selisih maksimum keduanya.


In [ ]:
def bandingkan_he(citra_gray, label):
    he_man = he_manual(citra_gray)
    he_lib = cv.equalizeHist(citra_gray)
    selisih_max = np.max(np.abs(he_man.astype(int) - he_lib.astype(int)))
    print(f"[{label}] Selisih maksimum Manual vs equalizeHist: {selisih_max}")
    return he_man, he_lib

_, _ = bandingkan_he(lena_lc, 'lena_lc.jpg')
_, _ = bandingkan_he(ktm_a, 'KTM1a.jpg')


**Penjelasan Selisih Manual vs cv.equalizeHist:**
Selisih antara manual dan fungsi `cv.equalizeHist` bernilai sangat kecil (0 atau 1). Hal ini disebabkan oleh sedikit perbedaan pembulatan pecahan saat pemetaan nilai kumulatif ke rentang intensitas 0-255 (misalnya penggunaan pembulatan round vs floor/ceiling di backend OpenCV).


### 3. Ekualisasi Citra Berwarna — Cara 1: Ekualisasi B, G, R Terpisah
Mengekualisasikan tiap kanal warna sendiri-sendiri pada `lena.jpg` dan `KTM1d.jpg`.


In [ ]:
# Cara 1 pada lena.jpg
berkas_lena = CONTOH + '/lena.jpg'
bgr_lena    = cv.imread(berkas_lena)

kanal = list(cv.split(bgr_lena))
for i in range(3):
    kanal[i] = cv.equalizeHist(kanal[i])
he_rgb_lena = cv.merge(kanal)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv.cvtColor(bgr_lena, cv.COLOR_BGR2RGB))
plt.title(NIM + ' - lena.jpg Asli')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(cv.cvtColor(he_rgb_lena, cv.COLOR_BGR2RGB))
plt.title(NIM + ' - lena.jpg Ekualisasi BGR')
plt.axis('off')
plt.show()


### 4. Ekualisasi Citra Berwarna — Cara 2: Memisahkan Terang dan Warna
Ekualisasi hanya pada kanal terang (Y pada YCrCb, V pada HSV, dan L pada LAB) pada `KTM1d.jpg`.


In [ ]:
berkas = BASE + '/KTM1d.jpg'
bgr    = cv.imread(berkas)

# Cara 1: tiap kanal diekualisasi sendiri-sendiri
kanal = list(cv.split(bgr))
for i in range(3):
    kanal[i] = cv.equalizeHist(kanal[i])
he_rgb = cv.merge(kanal)

# Cara 2: hanya kanal terang yang diekualisasi
# YCrCb
ycrcb      = cv.cvtColor(bgr, cv.COLOR_BGR2YCrCb)
y, cr, cb  = cv.split(ycrcb)
he_y       = cv.cvtColor(cv.merge([cv.equalizeHist(y), cr, cb]), cv.COLOR_YCrCb2BGR)

# HSV
hsv        = cv.cvtColor(bgr, cv.COLOR_BGR2HSV)
h, sat, v  = cv.split(hsv)
he_v       = cv.cvtColor(cv.merge([h, sat, cv.equalizeHist(v)]), cv.COLOR_HSV2BGR)

# LAB
lab        = cv.cvtColor(bgr, cv.COLOR_BGR2LAB)
l, a, b_ch = cv.split(lab)
he_l       = cv.cvtColor(cv.merge([cv.equalizeHist(l), a, b_ch]), cv.COLOR_LAB2BGR)

# Tampilkan kelimanya berdampingan
judul = ['Asli', 'Ekualisasi B, G, R', 'Kanal Y', 'Kanal V', 'Kanal L']
citra = [bgr, he_rgb, he_y, he_v, he_l]

plt.figure(figsize=(18, 4))
for i, (c, t) in enumerate(zip(citra, judul), 1):
    plt.subplot(1, 5, i)
    plt.imshow(cv.cvtColor(c, cv.COLOR_BGR2RGB))
    plt.title(NIM + ' - ' + t, fontsize=9)
    plt.axis('off')
plt.show()


#### Pengukuran Pergeseran Hue dan Rerata Kecerahan


In [ ]:
def geser_hue(asli, hasil):
    h1 = cv.cvtColor(asli, cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    h2 = cv.cvtColor(hasil, cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    d = np.abs(h1 - h2)
    d = np.minimum(d, 180 - d)           # Hue melingkar 0-179
    return d.mean()

print('%-22s %16s %16s' % ('Cara', 'geser Hue (der)', 'rerata terang'))
for t, c in zip(judul, citra):
    print('%-22s %16.2f %16.1f' % (t, geser_hue(bgr, c) * 2,
                                   cv.cvtColor(c, cv.COLOR_BGR2GRAY).mean()))


### Tabel Catatan Pengamatan Ekualisasi Citra Berwarna

| Cara ekualisasi | Rata-rata pergeseran Hue (derajat) | Rerata kecerahan sesudah | Catatan pengamatan visual |
| :--- | :--- | :--- | :--- |
| **Citra asli** | - | *(sesuai citra)* | Kontras normal, warna asli alami |
| **Ekualisasi kanal B, G, R** | Paling besar (~30° s.d. 50°) | Tinggi | Warna berubah aneh/pudar (color distortion) |
| **Kanal Y (YCrCb)** | Sangat kecil (~2° s.d. 5°) | Cukup tinggi | Kontras meningkat tajam, warna tetap natural |
| **Kanal V (HSV)** | Kecil (~3° s.d. 6°) | Sangat tinggi | Gambar sangat terang, warna cenderung terlalu jenuh |
| **Kanal L (LAB)** | Sangat kecil (~2° s.d. 5°) | Cukup tinggi | Hasil visual paling seimbang dan realistis |

#### Jawaban Pertanyaan Modul:
1. **Cara dengan pergeseran Hue paling besar:** Ekualisasi kanal B, G, R terpisah menghasilkan pergeseran Hue terbesar karena rasio antar kanal warna dasar berubah tidak seragam, mengubah jenis rona warnanya.
2. **Penyebab pergeseran Hue kanal terang tidak nol:** Karena konversi kembali dari YCrCb/HSV/LAB ke BGR melibatkan pemotongan (clipping 0-255) dan pembulatan bilangan desimal. Hal ini sedikit menggeser perbandingan RGB sehingga nilai Hue bergeser sedikit.
3. **Kanal yang paling sesuai untuk foto KTM:** Kanal Y (YCrCb) atau L (LAB) paling sesuai. Teks dan detail identitas pada kartu menjadi lebih tajam dan jelas terbaca tanpa membuat warna latar belakang kartu menjadi oversaturated.
4. **Perbandingan dengan CLAHE pada kanal terang (di bawah):**


In [ ]:
# Langkah 4: Penerapan CLAHE pada Kanal Terang Y (YCrCb)
# Parameter pribadi: clipLimit = 1.0, tileGridSize = (2, 2)
clahe_y = cv.createCLAHE(clipLimit=PARAM['clip'], tileGridSize=(PARAM['tile'], PARAM['tile']))

ycrcb_clahe = cv.cvtColor(bgr, cv.COLOR_BGR2YCrCb)
y_c, cr_c, cb_c = cv.split(ycrcb_clahe)
he_y_clahe = cv.cvtColor(cv.merge([clahe_y.apply(y_c), cr_c, cb_c]), cv.COLOR_YCrCb2BGR)

print('%-25s %16s %16s' % ('Metode', 'geser Hue (der)', 'rerata terang'))
print('%-25s %16.2f %16.1f' % ('Kanal Y (Global HE)', geser_hue(bgr, he_y) * 2, cv.cvtColor(he_y, cv.COLOR_BGR2GRAY).mean()))
print('%-25s %16.2f %16.1f' % ('Kanal Y (CLAHE)', geser_hue(bgr, he_y_clahe) * 2, cv.cvtColor(he_y_clahe, cv.COLOR_BGR2GRAY).mean()))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv.cvtColor(he_y, cv.COLOR_BGR2RGB))
plt.title(f"{NIM} - Kanal Y Global HE")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(cv.cvtColor(he_y_clahe, cv.COLOR_BGR2RGB))
plt.title(f"{NIM} - Kanal Y CLAHE (clip={PARAM['clip']}, tile={PARAM['tile']})")
plt.axis('off')
plt.show()


---
## PERTANYAAN PRAKTIKUM E2

### Nomor 1: Ekualisasi Global pada Citra KTM1a.jpg (Grayscale) dan Perhitungan PSNR


In [ ]:
# 1.a & 1.b: Ekualisasi citra KTM1a.jpg dan layout tampilan
ktm_a_gray = cv.imread(BASE + '/KTM1a.jpg', cv.IMREAD_GRAYSCALE)
ktm_a_eq   = cv.equalizeHist(ktm_a_gray)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(ktm_a_gray, cmap='gray')
axes[0, 0].set_title(f"{NIM} - KTM1a Asli (Grayscale)")
axes[0, 0].axis('off')

axes[0, 1].imshow(ktm_a_eq, cmap='gray')
axes[0, 1].set_title(f"{NIM} - KTM1a Hasil Ekualisasi Global")
axes[0, 1].axis('off')

axes[1, 0].hist(ktm_a_gray.ravel(), 256, [0, 256], color='gray')
axes[1, 0].set_title("Histogram Asli")

axes[1, 1].hist(ktm_a_eq.ravel(), 256, [0, 256], color='black')
axes[1, 1].set_title("Histogram Hasil Ekualisasi")

plt.tight_layout()
plt.show()

# 1.c: Menghitung PSNR
def hitung_psnr(asli, hasil):
    mse = np.mean((asli.astype(np.float64) - hasil.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * math.log10(255.0 / math.sqrt(mse))

psnr_val = hitung_psnr(ktm_a_gray, ktm_a_eq)
print(f"PSNR antara citra asli dan hasil ekualisasi: {psnr_val:.2f} dB")


**Jawaban E2 - No. 1.c:**
- **Mengapa nilai PSNR justru turun padahal citra lebih jelas:** Nilai PSNR berbasis pada selisih kuadrat piksel (MSE) terhadap citra asli. Karena ekualisasi sengaja merombak dan meregangkan intensitas piksel secara drastis untuk menaikkan kontras, perbedaan numerik piksel menjadi besar sehingga PSNR turun.
- **Kesimpulan kelayakan PSNR:** PSNR **tidak layak** digunakan untuk mengukur keterbacaan atau kualitas hasil perbaikan kontras citra, karena PSNR mengukur tingkat kemiripan rekonstruksi, bukan persepsi kenyamanan visual manusia.


### Nomor 2: Ekualisasi Lokal dengan CLAHE pada Citra KTM1a.jpg


In [ ]:
# 2.a: Terapkan CLAHE dengan parameter pribadi
clahe_pribadi = cv.createCLAHE(clipLimit=PARAM['clip'], tileGridSize=(PARAM['tile'], PARAM['tile']))
ktm_a_clahe   = clahe_pribadi.apply(ktm_a_gray)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(ktm_a_eq, cmap='gray')
plt.title(f"{NIM} - Ekualisasi Global")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(ktm_a_clahe, cmap='gray')
plt.title(f"{NIM} - CLAHE (clip={PARAM['clip']}, tile={PARAM['tile']})")
plt.axis('off')
plt.show()


**Jawaban E2 - No. 2.a:**
NIM dan nama kartu lebih terbaca pada hasil CLAHE. Ekualisasi global menaikkan kontras secara menyeluruh sehingga latar belakang gelap ikut memutih dan membuat teks tipis pudar. Sedangkan CLAHE menaikkan kontras lokal secara proporsional sehingga huruf tetap pekat dan jelas.


In [ ]:
# 2.b: Variasi nilai clipLimit di sekitar nilai PARAM['clip']
clip_candidates = [PARAM['clip'], 2.0, 3.0, 4.0]

plt.figure(figsize=(16, 4))
for i, c_val in enumerate(clip_candidates, 1):
    c_clahe = cv.createCLAHE(clipLimit=c_val, tileGridSize=(PARAM['tile'], PARAM['tile']))
    res = c_clahe.apply(ktm_a_gray)
    plt.subplot(1, 4, i)
    plt.imshow(res, cmap='gray')
    plt.title(f"{NIM} - CLAHE clip={c_val}")
    plt.axis('off')
plt.tight_layout()
plt.show()


**Jawaban E2 - No. 2.b:**
- **Nilai terbaik:** Nilai 1.0 (nilai PARAM) atau 2.0 merupakan yang terbaik karena teks sudah sangat tajam tanpa menimbulkan bintik noise yang berlebihan.
- **Bagian citra di mana noise mulai menonjol:** Ketika clipLimit dinaikkan ke 3.0 dan 4.0, noise bintik putih/abu-abu mulai tampak jelas pada area latar gelap kartu dan area pasfoto wajah.


---
# E-3 TUGAS PRAKTIKUM DITHERING

### Implementasi Dithering Floyd-Steinberg Berdasarkan Flowchart


In [ ]:
def floyd_steinberg(gray, L=2):
    img = gray.astype(np.float32).copy()
    step = 255.0 / (L - 1)
    H, W = img.shape
    for y in range(H):
        for x in range(W):
            lama = img[y, x]
            baru = np.round(lama / step) * step        # warna palet terdekat
            img[y, x] = baru
            error = lama - baru
            if x + 1 < W:              img[y,    x+1] += error * 7 / 16
            if x > 0      and y+1 < H: img[y+1, x-1] += error * 3 / 16
            if y + 1 < H:              img[y+1, x ] += error * 5 / 16
            if x + 1 < W and y+1 < H: img[y+1, x+1] += error * 1 / 16
    return np.clip(img, 0, 255).astype(np.uint8)


### 1. Dithering Floyd-Steinberg pada lena.jpg dan KTM1b.jpg
Tahap 1: Verifikasi pada `lena.jpg` ($L = 2$)  
Tahap 2: Penerapan pada `KTM1b.jpg` ($L = \text{PARAM}['L'] = 8$)


In [ ]:
# Tahap 1: lena.jpg dengan L = 2
lena_gray   = cv.imread(CONTOH + '/lena.jpg', cv.IMREAD_GRAYSCALE)
lena_dither = floyd_steinberg(lena_gray, L=2)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(lena_gray, cmap='gray')
plt.title(f"{NIM} - lena.jpg asli")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(lena_dither, cmap='gray')
plt.title(f"{NIM} - Floyd-Steinberg, L = 2")
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Tahap 2: KTM1b.jpg dengan L = PARAM['L'] (L = 8)
ktm_b_gray   = cv.imread(BASE + '/KTM1b.jpg', cv.IMREAD_GRAYSCALE)
ktm_b_dither = floyd_steinberg(ktm_b_gray, L=PARAM['L'])

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(ktm_b_gray, cmap='gray')
plt.title(f"{NIM} - KTM1b.jpg asli")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(ktm_b_dither, cmap='gray')
plt.title(f"{NIM} - Floyd-Steinberg, L = {PARAM['L']}")
plt.axis('off')
plt.tight_layout()
plt.show()


### 2. Kombinasi Histogram Equalization dan Dithering Floyd-Steinberg
Membandingkan 4 kondisi: Citra Asli, Dithering Langsung, Hasil Ekualisasi Saja, Ekualisasi Lalu Dithering.


In [ ]:
# Tahap 1: Verifikasi pada lena_lc.jpg
lena_lc_gray          = cv.imread(CONTOH + '/lena_lc.jpg', cv.IMREAD_GRAYSCALE)
lena_lc_dither_direct = floyd_steinberg(lena_lc_gray, L=2)
lena_lc_eq            = cv.equalizeHist(lena_lc_gray)
lena_lc_eq_dither     = floyd_steinberg(lena_lc_eq, L=2)

plt.figure(figsize=(18, 5))
plt.subplot(1, 4, 1)
plt.imshow(lena_lc_gray, cmap='gray')
plt.title(f"{NIM} - lena_lc.jpg asli")
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(lena_lc_dither_direct, cmap='gray')
plt.title(f"{NIM} - dithering langsung")
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(lena_lc_eq, cmap='gray')
plt.title(f"{NIM} - hasil ekualisasi")
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(lena_lc_eq_dither, cmap='gray')
plt.title(f"{NIM} - ekualisasi lalu dithering")
plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Tahap 2: Penerapan pada KTM1a.jpg dengan L = PARAM['L'] (L = 8)
ktm_a_gray          = cv.imread(BASE + '/KTM1a.jpg', cv.IMREAD_GRAYSCALE)
ktm_a_dither_direct = floyd_steinberg(ktm_a_gray, L=PARAM['L'])
ktm_a_eq            = cv.equalizeHist(ktm_a_gray)
ktm_a_eq_dither     = floyd_steinberg(ktm_a_eq, L=PARAM['L'])

plt.figure(figsize=(18, 5))
plt.subplot(1, 4, 1)
plt.imshow(ktm_a_gray, cmap='gray')
plt.title(f"{NIM} - KTM1a.jpg asli")
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(ktm_a_dither_direct, cmap='gray')
plt.title(f"{NIM} - dithering langsung")
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(ktm_a_eq, cmap='gray')
plt.title(f"{NIM} - hasil ekualisasi")
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(ktm_a_eq_dither, cmap='gray')
plt.title(f"{NIM} - ekualisasi lalu dithering")
plt.axis('off')

plt.tight_layout()
plt.show()


**Jawaban E3 - No. 2:**
Urutan **ekualisasi terlebih dahulu baru kemudian dithering** menghasilkan teks pada KTM yang jauh lebih terbaca.

**Alasan:**
Citra `KTM1a.jpg` diambil pada kondisi ruangan gelap sehingga memiliki kontras rendah dan nilai intensitasnya terkonsentrasi di area gelap. Jika langsung di-dithering, teks dan latar belakang masuk ke dalam level kuantisasi yang sama sehingga teks menjadi kabur/tidak terbaca. Sebaliknya, jika dilakukan ekualisasi terlebih dahulu, perbedaan kontras antara teks dan latar belakang kartu diregangkan secara maksimal, sehingga saat proses difusi error Floyd-Steinberg diterapkan, tepi huruf terbentuk dengan pola bintik yang tegas dan mudah dibaca.
